# Lab 7: HMM Noun Phrase Chunking (B-NP / I-NP / O)
Forward algorithm + Viterbi decoding, Add-K smoothing, evaluation with seqeval.

In [7]:
!pip install -q "setuptools<81" wheel
!pip install -q seqeval --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.8 MB/s eta 0:00:00:00:01
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


## 1. Load data

In [8]:
import numpy as np
from scipy.special import logsumexp

train_path = "/kaggle/input/datasets/mahithaupad/train-np-txt/train_NP.txt"
test_path  = "/kaggle/input/datasets/mahithaupad/test-np-txt/test_NP.txt"

def read_data(path):
    """One token per line: first column = word, last column = tag.
    Blank line separates sentences."""
    sents, cur = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if cur:
                    sents.append(cur)
                    cur = []
                continue
            parts = line.split()
            cur.append((parts[0], parts[-1]))
    if cur:
        sents.append(cur)
    return sents

train = read_data(train_path)
test  = read_data(test_path)
print("train sentences:", len(train), "| test sentences:", len(test))

tags   = sorted({t for s in train for _, t in s})
tag2i  = {t: i for i, t in enumerate(tags)}
vocab  = sorted({w for s in train for w, _ in s})
word2i = {w: i for i, w in enumerate(vocab)}
UNK = len(word2i)                 # index reserved for unseen words
T, V = len(tags), len(word2i) + 1
print("tags:", tags, "| vocab size:", len(vocab))

train sentences: 823 | test sentences: 77
tags: ['B-NP', 'I-NP', 'O'] | vocab size: 4570


## 2. Train the HMM (counts + Add-K smoothing)

In [ ]:
start_c = np.zeros(T)
trans_c = np.zeros((T, T))
emit_c  = np.zeros((T, V))

for s in train:
    start_c[tag2i[s[0][1]]] += 1
    for j, (w, t) in enumerate(s):
        emit_c[tag2i[t], word2i[w]] += 1
        if j > 0:
            trans_c[tag2i[s[j-1][1]], tag2i[t]] += 1

def build_model(k):
    """Add-K smoothed log probabilities."""
    log_pi = np.log((start_c + k) / (start_c.sum() + k * T))
    log_A  = np.log((trans_c + k) / (trans_c.sum(1, keepdims=True) + k * T))
    log_B  = np.log((emit_c  + k) / (emit_c.sum(1, keepdims=True)  + k * V))
    return log_pi, log_A, log_B

## 3. Task 1: Forward algorithm (sentence probability)

In [10]:
def obs_idx(words):
    return [word2i.get(w, UNK) for w in words]

def forward_logprob(words, model):
    log_pi, log_A, log_B = model
    o = obs_idx(words)
    alpha = log_pi + log_B[:, o[0]]
    for t in range(1, len(o)):
        alpha = logsumexp(alpha[:, None] + log_A, axis=0) + log_B[:, o[t]]
    return logsumexp(alpha)       # log P(sentence)

## 4. Task 2: Viterbi decoding

In [11]:
def viterbi(words, model):
    log_pi, log_A, log_B = model
    o = obs_idx(words)
    n = len(o)
    delta = np.zeros((n, T))
    back  = np.zeros((n, T), dtype=int)
    delta[0] = log_pi + log_B[:, o[0]]
    for t in range(1, n):
        scores = delta[t-1][:, None] + log_A
        back[t]  = scores.argmax(0)
        delta[t] = scores.max(0) + log_B[:, o[t]]
    path = [int(delta[-1].argmax())]
    for t in range(n-1, 0, -1):
        path.append(int(back[t][path[-1]]))
    return [tags[i] for i in reversed(path)]

## Evaluation metrics (seqeval, with automatic fallback)

In [12]:
try:
    from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
    print("Using the real seqeval library")
except ImportError:
    print("seqeval not available, using equivalent built-in implementation")

    def _chunks(seq):
        out, start = set(), None
        for i, t in enumerate(list(seq) + ["O"]):
            if t == "O" or t.startswith("B-"):
                if start is not None:
                    out.add((start, i))
                    start = None
                if t.startswith("B-"):
                    start = i
            elif start is None:          # I- tag with no open chunk starts one
                start = i
        return out

    def _counts(y_true, y_pred):
        tp = fp = fn = 0
        for a, b in zip(y_true, y_pred):
            ta, pb = _chunks(a), _chunks(b)
            tp += len(ta & pb)
            fp += len(pb - ta)
            fn += len(ta - pb)
        return tp, fp, fn

    def precision_score(y_true, y_pred):
        tp, fp, fn = _counts(y_true, y_pred)
        return tp / (tp + fp) if tp + fp else 0.0

    def recall_score(y_true, y_pred):
        tp, fp, fn = _counts(y_true, y_pred)
        return tp / (tp + fn) if tp + fn else 0.0

    def f1_score(y_true, y_pred):
        p, r = precision_score(y_true, y_pred), recall_score(y_true, y_pred)
        return 2 * p * r / (p + r) if p + r else 0.0

    def classification_report(y_true, y_pred):
        tp, fp, fn = _counts(y_true, y_pred)
        return (f"{'':>8}{'precision':>10}{'recall':>10}{'f1-score':>10}{'support':>9}\n"
                f"{'NP':>8}{precision_score(y_true, y_pred):>10.4f}"
                f"{recall_score(y_true, y_pred):>10.4f}"
                f"{f1_score(y_true, y_pred):>10.4f}{tp + fn:>9}")

seqeval not available, using equivalent built-in implementation


## 5. Evaluate for different K (seqeval)

In [13]:
y_true = [[t for _, t in s] for s in test]
test_words = [[w for w, _ in s] for s in test]

results = {}
for k in [1, 0.1, 0.01, 0.001]:
    model = build_model(k)
    y_pred = [viterbi(ws, model) for ws in test_words]
    results[k] = (f1_score(y_true, y_pred),
                  precision_score(y_true, y_pred),
                  recall_score(y_true, y_pred),
                  y_pred)

print(f"{'K':>8} {'Precision':>10} {'Recall':>10} {'F1':>10}")
for k, (f1, p, r, _) in results.items():
    print(f"{k:>8} {p:>10.4f} {r:>10.4f} {f1:>10.4f}")

best_k = max(results, key=lambda k: results[k][0])
print("\nBest K:", best_k)
best_pred = results[best_k][3]
print(classification_report(y_true, best_pred))

       K  Precision     Recall         F1
       1     0.6774     0.6632     0.6702
     0.1     0.6716     0.7579     0.7122
    0.01     0.6381     0.7200     0.6766
   0.001     0.6292     0.7074     0.6660

Best K: 0.1
         precision    recall  f1-score  support
      NP    0.6716    0.7579    0.7122      475


## 6. Sentence probabilities from the forward algorithm (best K)

In [14]:
import pandas as pd

best_model = build_model(best_k)
log_probs = [forward_logprob(ws, best_model) for ws in test_words]

df = pd.DataFrame({
    "sentence": [" ".join(ws) for ws in test_words],
    "length": [len(ws) for ws in test_words],
    "log_prob": log_probs,
    "prob": np.exp(log_probs),     # may underflow to 0 for long sentences
})
print(df[["length", "log_prob", "prob"]].head(10))
df.to_csv("/kaggle/working/test_sentence_probs.csv", index=False)

for i in range(3):
    print("\nWords:", test_words[i])
    print("True: ", y_true[i])
    print("Pred: ", best_pred[i])

   length    log_prob           prob
0      37 -271.508725  1.216911e-118
1      27 -197.992415   1.030354e-86
2      29 -221.496125   6.389330e-97
3      36 -242.698930  3.955430e-106
4      31 -229.189333  2.912979e-100
5      36 -260.042134  1.161813e-113
6      39 -275.248866  2.890256e-120
7      25 -186.036478   1.604664e-81
8      28 -211.305064   1.703635e-92
9      24 -176.620095   1.971820e-77

Words: ['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'September', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'July', 'and', 'August', "'s", 'near-record', 'deficits', '.']
True:  ['B-NP', 'O', 'B-NP', 'I-NP', 'O', 'O', 'O', 'O', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'O', 'B-NP', 'O', 'O', 'O', 'B-NP', 'B-NP', 'O', 'O', 'O', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'B-NP', 'I-NP', 'I-NP', 'O']
Pred